## Analyse : Quels sont les éléments naturels d'une voie déterminants dans sa cotation ?

## 1. Import des librairies

In [1]:
import os
os.environ["PYTENSOR_FLAGS"] = "cxx=g++ -fexhaustive-register-search"
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import re
import unicodedata
import bambi as bmb
import arviz as az
import h5netcdf

In [2]:
import pytensor
print(pytensor.config.cxx)   

g++ -fexhaustive-register-search


## 2. Import des données

In [3]:
df = pd.read_csv("annotation_finale_ponderee.csv")
df

C:\Users\lunag\AppData\Local\Temp\ipykernel_13592\1127046363.py:1: DtypeWarning: Columns (31) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("annotation_finale_ponderee.csv")


,document_id,title,title_prefix,quality,activities,climbing_outdoor_type,global_rating,engagement_rating,risk_rating,equipment_rating,...,arête_ponderation,couloir_ponderation,trou_ponderation,réglette_ponderation,cannelure_ponderation,goutte_eau_ponderation,plat_ponderation,bac_ponderation,éperon_ponderation,crochet_ponderation
0,1948176,Triangle des Coutchets - Soleil d'automne,Pic des Coutchets,medium,rock_climbing,multi,TD-,NaN,NaN,P2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1948176,Triangle des Coutchets - Soleil d'automne,Pic des Coutchets,medium,rock_climbing,multi,TD-,NaN,NaN,P2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1948176,Triangle des Coutchets - Soleil d'automne,Pic des Coutchets,medium,rock_climbing,multi,TD-,NaN,NaN,P2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1948176,Triangle des Coutchets - Soleil d'automne,Pic des Coutchets,medium,rock_climbing,multi,TD-,NaN,NaN,P2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1944609,Capricho,Aiguille de la Canalona,medium,rock_climbing,multi,D+,NaN,NaN,P2+,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42856,53893,Voie Directe,Miroir d'Argentine,fine,rock_climbing,multi,D,II,NaN,P2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
42857,53893,Voie Directe,Miroir d'Argentine,fine,rock_climbing,multi,D,II,NaN,P2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
42858,53893,Voie Directe,Miroir d'Argentine,fine,rock_climbing,multi,D,II,NaN,P2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
42859,53893,Voie Directe,Miroir d'Argentine,fine,rock_climbing,multi,D,II,NaN,P2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 3. Manipulation des données

In [4]:
ordre_cotations = [
    "1a", "1b", "1b+", "1c+", 
    "2a", "2a+", "2b", "2b+", "2c", "2c+", 
    "3a", "3a+", "3b", "3b+", "3c", "3c+", 
    "4a", "4a+", "4b", "4b+", "4c", "4c+", 
    "5a", "5a+", "5b", "5b+", "5c", "5c+", 
    "6a", "6a+", "6b", "6b+", "6c", "6c+",
    "7a", "7a+", "7b", "7b+", "7c", "7c+", 
    "8a", "8a+", "8b", "8b+", "8c", "8c+",
    "9a+"
]

In [5]:
# Ordre des cotations

df["difficulty"] = pd.Categorical(
    df["difficulty"],
    categories=ordre_cotations,
    ordered=True
)

In [6]:
# Fréquents : gardés en compte numérique, plafonnés à 3
elements_frequents = [
    "dalle", "fissure", "dièdre", "traversée", "surplomb",
    "dévers", "cheminée", "vire", "arête", "pilier", "terrasse", "éperon",
]

# Intermédiaires : binarisés en présence/absence
elements_intermediaires = [
    "écailles", "trou", "couloir", "cannelure", "réglette",
    "goutte_eau", "crochet", "bac", "plat", "petite_prise", "faille",
]

# Ultra-rares : exclus du modèle
elements_rares = [
    "pince", "mono_doigt", "bi_doigt", "contrefort", "corniche",
    "épaule", "inversee", "fissure-dièdre",
]

# --- Construction du dataset d'analyse (df reste intact) ---
df_model = df.copy()

# 1. Fréquents : compte numérique plafonné à 3
for e in elements_frequents:
    df_model[f"nb_{e}"] = pd.to_numeric(df_model[f"nb_{e}"]).clip(upper=3)

# 2. Intermédiaires : binarisation en pres_<element>
for e in elements_intermediaires:
    df_model[f"pres_{e}"] = (pd.to_numeric(df_model[f"nb_{e}"]) > 0).astype(int)

# 3. Sélection des colonnes finales : variables du modèle + identifiants
colonnes_modele = (
    [f"nb_{e}"  for e in elements_frequents] +
    [f"pres_{e}" for e in elements_intermediaires]
)
df_model = df_model[colonnes_modele + ["difficulty", "document_id", 'log_nb_terms_description']].copy()

# 4. Suppression des lignes incomplètes (s'il y a des NaN résiduels)
df_model = df_model.dropna(subset=colonnes_modele + ["difficulty"]).reset_index(drop=True)

## 4. Analyse

In [7]:
# ============================================================
# PARAMÈTRES À ADAPTER
# ============================================================
COL_COTATION = "difficulty"                 
COL_VOIE = "document_id"
PREDICTEURS = ['nb_dalle',
 'nb_fissure',
 'nb_dièdre',
 'nb_traversée',
 'nb_surplomb',
 'nb_dévers',
 'nb_cheminée',
 'nb_vire',
 'nb_arête',
 'nb_pilier',
 'nb_terrasse',
 'nb_éperon',
 'pres_écailles',
 'pres_trou',
 'pres_couloir',
 'pres_cannelure',
 'pres_réglette',
 'pres_goutte_eau',
 'pres_crochet',
 'pres_bac',
 'pres_plat',
 'pres_petite_prise',
 'pres_faille',
 'log_nb_terms_description']   

# Classes de cotation : on borne les extrêmes (rares) -> ici 15 classes
COTATION_MIN = "4c+"                        # tout ce qui est <= devient la classe 1
COTATION_MAX = "7a+"                        # tout ce qui est >= devient la dernière classe

SEPARER_ENTRE_INTRA = True                 # True : effet "à voie égale" et effet "entre voies" séparés
N_VOIES_TEST = None                         # ex. 300 pour un essai rapide ; None = tout

DOSSIER = "resultats"
PREFIXE = os.path.join(DOSSIER, "ordinaleA_Total_bambi")

In [8]:
# ============================================================
# OUTILS
# ============================================================
def nettoyer_nom(nom):
    """Nom de colonne sans accent ni caractère spécial (nécessaire pour les formules bambi)."""
    s = unicodedata.normalize("NFKD", nom).encode("ascii", "ignore").decode()
    return re.sub(r"\W+", "_", s).strip("_")


ORDRE = [f"{n}{l}{p}" for n in range(1, 10) for l in "abc" for p in ["", "+"]]
RANG = {g: i for i, g in enumerate(ORDRE)}


def cotation_vers_classe(serie):
    """Cotation -> entier croissant (1 = plus facile), extrêmes regroupés."""
    rang = serie.astype("string").str.strip().str.lower().map(RANG).astype(float)
    bas, haut = RANG[COTATION_MIN], RANG[COTATION_MAX]
    return rang.clip(lower=bas, upper=haut) - bas + 1

In [9]:
# ============================================================
# 1. PRÉPARATION DES DONNÉES
# ============================================================

data = pd.DataFrame({
    "classe_cotation": cotation_vers_classe(df_model[COL_COTATION]),
    COL_VOIE: df_model[COL_VOIE],
})
noms_propres = {p: nettoyer_nom(p) for p in PREDICTEURS}
for p, propre in noms_propres.items():
    data[propre] = pd.to_numeric(df_model[p], errors="coerce")
predicteurs = list(noms_propres.values())

data = data.dropna(subset=["classe_cotation"] + predicteurs).copy()
data["classe_cotation"] = data["classe_cotation"].astype(int)
print(f"{len(data)} lignes, {data[COL_VOIE].nunique()} voies, "
      f"{data['classe_cotation'].nunique()} classes de cotation")

if N_VOIES_TEST:
    voies = data[COL_VOIE].drop_duplicates().sample(N_VOIES_TEST, random_state=0)
    data = data[data[COL_VOIE].isin(voies)].copy()
    print(f"Essai rapide : {data[COL_VOIE].nunique()} voies, {len(data)} lignes")

if SEPARER_ENTRE_INTRA:
    nouvelles = []
    for p in predicteurs:
        moyenne = data.groupby(COL_VOIE)[p].transform("mean")
        data[p + "_moy_voie"] = moyenne          # effet "entre voies"
        data[p + "_ecart"] = data[p] - moyenne   # effet "à voie égale"
        nouvelles += [p + "_ecart", p + "_moy_voie"]
    predicteurs = nouvelles

SEUIL_FUSION = 20 if N_VOIES_TEST else 400
effectifs = data["classe_cotation"].value_counts().sort_index()
mapping = {}
for c in effectifs.index:
    if effectifs[c] < SEUIL_FUSION and (c - 1) in effectifs.index:
        mapping[c] = c - 1      # fusion dans la classe inférieure
    else:
        mapping[c] = c
data["classe_cotation"] = data["classe_cotation"].map(mapping)
print("Effectifs des classes après fusion :")
print(data["classe_cotation"].value_counts().sort_index())

os.makedirs(DOSSIER, exist_ok=True)
data.to_csv(os.path.join(DOSSIER, "donnees_prepareesA.csv"), index=False)

42861 lignes, 9299 voies, 15 classes de cotation
Effectifs des classes après fusion :
classe_cotation
1     5747
2     2358
4     4245
5      521
6     5996
7     1064
8     6303
9     4084
10    4144
11    2424
12    2278
13    1054
14    1121
15    1522
Name: count, dtype: int64


In [ ]:
exclues = {"classe_cotation", COL_VOIE}
predicteurs_modele = nouvelles    # uniquement les _ecart/_moy_voie (48), pas les bruts

# 1) Données : standardisation + winsorisation (comme le faisait le script R)
df_bayes = data.copy()
for p in predicteurs_modele:
    x = df_bayes[p].astype(float)
    z = (x - x.mean()) / x.std()
    df_bayes[p] = z.clip(-10, 10)          # winsorisation à ±10

# 2) Réponse = catégorie ORDONNÉE (indispensable pour la famille cumulative)
categories = sorted(df_bayes["classe_cotation"].unique())
df_bayes["classe_cotation"] = pd.Categorical(df_bayes["classe_cotation"],
                                             categories=categories, ordered=True)
# 3) La voie doit être un facteur, pas un nombre
df_bayes[COL_VOIE] = df_bayes[COL_VOIE].astype("category")

# 4) Priors faiblement informatifs
priors = {p: bmb.Prior("Normal", mu=0, sigma=2) for p in predicteurs_modele}
priors["group_specific"] = bmb.Prior(
    "Normal", mu=0, sigma=bmb.Prior("HalfNormal", sigma=2.5)
)
formule = ("classe_cotation ~ " + " + ".join(predicteurs_modele)
           + f" + (1|{COL_VOIE})")
print(formule)

model = bmb.Model(formule, df_bayes, family="cumulative", priors=priors)
print(model)

# 5) Run complet
idata = model.fit(
    draws=1500,
    tune=1000,
    chains=4,
    cores=4,
    inference_method="nutpie",
    random_seed=42,
    progressbar=False,
)

idata.to_netcdf("idata_A.nc")   

# 6) DIAGNOSTICS — les critères de validation
resume = az.summary(idata, ci_prob=0.95, round_to=3)
print(resume)
# À vérifier : r_hat ≈ 1.00 partout (max < 1.01) et ess_bulk > 400
print("Rhat max :", resume["r_hat"].max())

# 7) Export CSV des résultats
post = az.summary(idata, var_names=predicteurs_modele, ci_prob=0.95)
res = pd.DataFrame({
    "variable":   post.index,
    "coef":       post["mean"],
    "erreur_std": post["sd"],
    "ic_bas":     post["eti95_lb"],
    "ic_haut":    post["eti95_ub"],
})
res["odds_ratio"] = np.exp(res["coef"])
res.to_csv(f"{PREFIXE}_coefficients.csv", index=False)

sigma_voie = float(az.summary(idata, var_names=[f"1|{COL_VOIE}_sigma"])["mean"].iloc[0])

pd.DataFrame({"n_obs": [len(df_bayes)],
              "n_voies": [df_bayes[COL_VOIE].nunique()],
              "n_classes": [df_bayes["classe_cotation"].nunique()],
              "sd_alea": [sigma_voie],
              "variance_alea": [sigma_voie**2]}
             ).to_csv(f"{PREFIXE}_ajustement.csv", index=False)
print("Résultats écrits :", PREFIXE, "_*.csv")

classe_cotation ~ nb_dalle_ecart + nb_dalle_moy_voie + nb_fissure_ecart + nb_fissure_moy_voie + nb_diedre_ecart + nb_diedre_moy_voie + nb_traversee_ecart + nb_traversee_moy_voie + nb_surplomb_ecart + nb_surplomb_moy_voie + nb_devers_ecart + nb_devers_moy_voie + nb_cheminee_ecart + nb_cheminee_moy_voie + nb_vire_ecart + nb_vire_moy_voie + nb_arete_ecart + nb_arete_moy_voie + nb_pilier_ecart + nb_pilier_moy_voie + nb_terrasse_ecart + nb_terrasse_moy_voie + nb_eperon_ecart + nb_eperon_moy_voie + pres_ecailles_ecart + pres_ecailles_moy_voie + pres_trou_ecart + pres_trou_moy_voie + pres_couloir_ecart + pres_couloir_moy_voie + pres_cannelure_ecart + pres_cannelure_moy_voie + pres_reglette_ecart + pres_reglette_moy_voie + pres_goutte_eau_ecart + pres_goutte_eau_moy_voie + pres_crochet_ecart + pres_crochet_moy_voie + pres_bac_ecart + pres_bac_moy_voie + pres_plat_ecart + pres_plat_moy_voie + pres_petite_prise_ecart + pres_petite_prise_moy_voie + pres_faille_ecart + pres_faille_moy_voie + log_n

c:\Python313\Lib\site-packages\bambi\formula.py:112: UserWarning: The intercept is omitted in ordinal families
  warnings.warn("The intercept is omitted in ordinal families")


       Formula: classe_cotation ~ nb_dalle_ecart + nb_dalle_moy_voie + nb_fissure_ecart + nb_fissure_moy_voie + nb_diedre_ecart + nb_diedre_moy_voie + nb_traversee_ecart + nb_traversee_moy_voie + nb_surplomb_ecart + nb_surplomb_moy_voie + nb_devers_ecart + nb_devers_moy_voie + nb_cheminee_ecart + nb_cheminee_moy_voie + nb_vire_ecart + nb_vire_moy_voie + nb_arete_ecart + nb_arete_moy_voie + nb_pilier_ecart + nb_pilier_moy_voie + nb_terrasse_ecart + nb_terrasse_moy_voie + nb_eperon_ecart + nb_eperon_moy_voie + pres_ecailles_ecart + pres_ecailles_moy_voie + pres_trou_ecart + pres_trou_moy_voie + pres_couloir_ecart + pres_couloir_moy_voie + pres_cannelure_ecart + pres_cannelure_moy_voie + pres_reglette_ecart + pres_reglette_moy_voie + pres_goutte_eau_ecart + pres_goutte_eau_moy_voie + pres_crochet_ecart + pres_crochet_moy_voie + pres_bac_ecart + pres_bac_moy_voie + pres_plat_ecart + pres_plat_moy_voie + pres_petite_prise_ecart + pres_petite_prise_moy_voie + pres_faille_ecart + pres_faille_

NUTS[nutpie]: [threshold, nb_dalle_ecart, nb_dalle_moy_voie, nb_fissure_ecart, nb_fissure_moy_voie, nb_diedre_ecart, nb_diedre_moy_voie, nb_traversee_ecart, nb_traversee_moy_voie, nb_surplomb_ecart, nb_surplomb_moy_voie, nb_devers_ecart, nb_devers_moy_voie, nb_cheminee_ecart, nb_cheminee_moy_voie, nb_vire_ecart, nb_vire_moy_voie, nb_arete_ecart, nb_arete_moy_voie, nb_pilier_ecart, nb_pilier_moy_voie, nb_terrasse_ecart, nb_terrasse_moy_voie, nb_eperon_ecart, nb_eperon_moy_voie, pres_ecailles_ecart, pres_ecailles_moy_voie, pres_trou_ecart, pres_trou_moy_voie, pres_couloir_ecart, pres_couloir_moy_voie, pres_cannelure_ecart, pres_cannelure_moy_voie, pres_reglette_ecart, pres_reglette_moy_voie, pres_goutte_eau_ecart, pres_goutte_eau_moy_voie, pres_crochet_ecart, pres_crochet_moy_voie, pres_bac_ecart, pres_bac_moy_voie, pres_plat_ecart, pres_plat_moy_voie, pres_petite_prise_ecart, pres_petite_prise_moy_voie, pres_faille_ecart, pres_faille_moy_voie, log_nb_terms_description_ecart, log_nb_term

In [ ]:
# ============================================================
# 3. RÉSULTATS
# ============================================================
coefs = pd.read_csv(PREFIXE + "_coefficients.csv")
ajust = pd.read_csv(PREFIXE + "_ajustement.csv")
print(ajust.to_string(index=False))
print(coefs.round(3).to_string(index=False))

# Forest plot (coefficients ; positif = cotation plus haute)
coefs = coefs.sort_values("coef").reset_index(drop=True)
y_pos = range(1, len(coefs) + 1)
ci = (coefs["ic_haut"] - coefs["ic_bas"]) / 2

plt.figure(figsize=(8, 0.4 * len(coefs) + 2))
plt.errorbar(coefs["coef"], y_pos, xerr=ci, fmt="o", ecolor="purple")
plt.yticks(y_pos, coefs["variable"], fontsize=16)
plt.axvline(x=0, linestyle="--", color="red")
plt.xlabel("Coefficients", fontsize=15)
plt.title("Model A", fontsize=15)
plt.grid(axis="x")

FileNotFoundError: [Errno 2] No such file or directory: 'resultats\\ordinaleA_Total_bambi_coefficients.csv'